# 08 — RQ1 Primary Inference: Median log Token Premium

**Protocol**: `NB08_RQ1_PROTOCOL_v001` · **Decision**: `RD-RQ1-FIRST-RESULT-01`

| record | SHA |
|---|---|
| base main | `79490b723ff4413763a84d310e50fa2748ccca6c` |
| decision | `e72274086a7e9c611c9014e6b5612df0e69dae30` |
| cohort | `9b695307c0551be84d4d6c374646bfe001b7b3a9` |
| protocol | `86521fdf04839d2e3e8e5db8e15a08ea067871e3` |

All three were committed **before** any result was observed.

**RQ1** — Under the fixed `o200k_base` Track A measurement and the final KO–EN semantically matched
cohort, is `Median(log Tokenization Premium) > 0`?

Every quantity below is computed from the D-04 physical artifact. No value is copied from EDA V1 or
EDA V2; EDA V2 is not an inference source. No raw KO/EN text is loaded or emitted.

Out of scope for this notebook: NB06, regex chunking, D-05, morphology explanatory models, M0–M3,
VIF/condition number, source_domain modeling, near-duplicate clustering, train/test split, NB10,
G-ID, causal claims.

## 01 — Canonical D-04 fail-closed validation

In [1]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
import platform
import time
from pathlib import Path
from zoneinfo import ZoneInfo

import duckdb
import numpy as np
import pyarrow
import pyarrow.parquet as pq
import scipy
import scipy.stats as st

KST = ZoneInfo("Asia/Seoul")
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

D04 = ROOT / "data/registry/TOKEN_O200K_BASE_v001.parquet"
D04_SHA256 = "1c30e3276222dd94885ae4f79fc6ab5c45e4e26226de0afd91fc6b1f7d2c16e7"
PAIR_SET_HASH = "d9660d654ee449e4d0c23a0070225274"
EXPECTED_N = 3_835_988

BASE_MAIN_SHA = "79490b723ff4413763a84d310e50fa2748ccca6c"
RQ1_DECISION_SHA = "e72274086a7e9c611c9014e6b5612df0e69dae30"
RQ1_COHORT_SHA = "9b695307c0551be84d4d6c374646bfe001b7b3a9"
RQ1_PROTOCOL_SHA = "86521fdf04839d2e3e8e5db8e15a08ea067871e3"

BOOTSTRAP_B = 2000
BOOTSTRAP_SEED = 969634713
BOOTSTRAP_SEED_SOURCE = "RD-RQ1-FIRST-RESULT-01|NB08_RQ1_PROTOCOL_v001"
CI_Q = (0.025, 0.975)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 22), b""):
            h.update(chunk)
    return h.hexdigest()


assert hashlib.sha256(BOOTSTRAP_SEED_SOURCE.encode()).hexdigest()[:8] == f"{BOOTSTRAP_SEED:08x}", (
    "seed does not reproduce from its declared source string"
)

actual_sha = sha256_file(D04)
if actual_sha != D04_SHA256:
    raise SystemExit(f"CANONICAL_ARTIFACT_IDENTITY_MISMATCH: {actual_sha}")

schema = pq.read_schema(D04)
meta = pq.ParquetFile(D04).metadata
print(f"D-04 sha256    {actual_sha}  MATCH")
print(f"rows           {meta.num_rows:,}")
print(f"columns        {len(schema.names)}")
print(f"row groups     {meta.num_row_groups}")

D-04 sha256    1c30e3276222dd94885ae4f79fc6ab5c45e4e26226de0afd91fc6b1f7d2c16e7  MATCH
rows           3,835,988
columns        28
row groups     1535


## 02 — RQ1 cohort manifest validation

In [2]:
COHORT_MANIFEST = json.loads((ROOT / "ssot_nb01/02_ANALYSIS_COHORT_RQ1_v001.json").read_text())

con = duckdb.connect()
con.execute("SET memory_limit='5GB'")
con.execute("SET threads=8")
con.execute("SET preserve_insertion_order=false")
REL = f"read_parquet('{D04.as_posix()}')"

n_rows, n_distinct, n_null_id, n_null_y, n_nonfinite = con.execute(
    f"""SELECT count(*), count(DISTINCT pair_id),
              sum((pair_id IS NULL)::INT),
              sum((log_token_premium IS NULL)::INT),
              sum((NOT isfinite(log_token_premium))::INT)
       FROM {REL}"""
).fetchone()
pair_set_hash = con.execute(
    f"SELECT md5(string_agg(pair_id,'' ORDER BY pair_id)) FROM {REL}"
).fetchone()[0]

checks = {
    "row_count": (n_rows, EXPECTED_N),
    "distinct_pair_id": (n_distinct, EXPECTED_N),
    "null_pair_id": (n_null_id, 0),
    "null_outcome": (n_null_y, 0),
    "nonfinite_outcome": (n_nonfinite, 0),
    "pair_set_hash": (pair_set_hash, PAIR_SET_HASH),
    "manifest_row_count": (COHORT_MANIFEST["row_count"], EXPECTED_N),
    "manifest_status": (COHORT_MANIFEST["validation_status"], "PASS"),
}
for name, (got, want) in checks.items():
    status = "OK" if got == want else "FAIL"
    print(f"  {name:22s} {str(got):36s} {status}")
if any(got != want for got, want in checks.values()):
    raise SystemExit("RQ1_COHORT_VALIDATION_FAILED")

# Protocol section 7: hard fail on non-finite. No post-hoc deletion is permitted.
if n_nonfinite != 0:
    raise SystemExit("NONFINITE_OUTCOME_HARD_FAIL")
print("\nRQ1_COHORT_VALIDATED")

  row_count              3835988                              OK
  distinct_pair_id       3835988                              OK
  null_pair_id           0                                    OK
  null_outcome           0                                    OK
  nonfinite_outcome      0                                    OK
  pair_set_hash          d9660d654ee449e4d0c23a0070225274     OK
  manifest_row_count     3835988                              OK
  manifest_status        PASS                                 OK

RQ1_COHORT_VALIDATED


## 03 — Descriptive outcome snapshot

The outcome column is materialised once. Only `log_token_premium` and the direction stratum are
loaded — no token ID arrays and no text.

In [3]:
t0 = time.monotonic()
tbl = con.execute(
    f"""SELECT t.log_token_premium AS y,
              (p.translation_direction IS DISTINCT FROM 'UNKNOWN') AS known_dir
       FROM {REL} t
       JOIN read_parquet('{(ROOT / 'data/registry/PAIR_REGISTRY_v002.parquet').as_posix()}') p
         USING (pair_id)"""
).fetch_arrow_table()
Y = tbl.column("y").to_numpy(zero_copy_only=False).astype(np.float64)
KNOWN = tbl.column("known_dir").to_numpy(zero_copy_only=False).astype(bool)
del tbl
load_sec = round(time.monotonic() - t0, 2)

assert Y.shape[0] == EXPECTED_N, Y.shape
assert np.isfinite(Y).all(), "NONFINITE_OUTCOME_HARD_FAIL"

n_known = int(KNOWN.sum())
print(f"loaded {Y.size:,} outcomes in {load_sec}s")
print(f"known-direction rows {n_known:,}   unknown {Y.size - n_known:,}")

desc = {
    "n": int(Y.size),
    "mean": float(Y.mean()),
    "sd": float(Y.std(ddof=1)),
    "min": float(Y.min()),
    "p01": float(np.percentile(Y, 1)),
    "p25": float(np.percentile(Y, 25)),
    "median": float(np.median(Y)),
    "p75": float(np.percentile(Y, 75)),
    "p99": float(np.percentile(Y, 99)),
    "max": float(Y.max()),
    "n_zero_exact": int((Y == 0.0).sum()),
    "n_positive": int((Y > 0.0).sum()),
    "n_negative": int((Y < 0.0).sum()),
}
desc["share_TP_gt_1"] = desc["n_positive"] / desc["n"]
for k, v in desc.items():
    print(f"  {k:16s} {v}")

/tmp/ipykernel_240542/2377813471.py:8: DeprecationWarning: fetch_arrow_table() is deprecated, use to_arrow_table() instead.
  ).fetch_arrow_table()


loaded 3,835,988 outcomes in 0.76s
known-direction rows 3,785,441   unknown 50,547
  n                3835988
  mean             0.28517678624044906
  sd               0.22209552728201393
  min              -2.74859629549656
  p01              -0.2876820724517809
  p25              0.15415067982725836
  median           0.28768207245178085
  p75              0.42744401482693967
  p99              0.8109302162163288
  max              3.6375861597263857
  n_zero_exact     196718
  n_positive       3375095
  n_negative       264175
  share_TP_gt_1    0.8798502497922308


## 04 — Wilcoxon signed-rank (primary test)

Frozen settings: `alternative='greater'`, `zero_method='wilcox'` (exact zeros dropped).
The p-value is never reported as 0; on underflow `log10(p)` is derived from the normal
approximation.

In [4]:
def log10_p_from_normal(z: float) -> float:
    """log10 upper-tail probability of the standard normal, underflow-safe."""
    return float(st.norm.logsf(z) / np.log(10.0))


def wilcoxon_report(y: np.ndarray, label: str) -> dict:
    n_zero = int((y == 0.0).sum())
    res = st.wilcoxon(y, alternative="greater", zero_method="wilcox")
    nz = y[y != 0.0]
    n_eff = nz.size
    # Normal-approximation z for underflow-safe log10(p), matching zero_method='wilcox'.
    ranks = st.rankdata(np.abs(nz))
    w_plus = float(ranks[nz > 0].sum())
    mu = n_eff * (n_eff + 1) / 4.0
    # tie correction on absolute-value ranks
    _, counts = np.unique(np.abs(nz), return_counts=True)
    tie_term = float(((counts ** 3 - counts).sum()) / 48.0)
    sigma = float(np.sqrt(n_eff * (n_eff + 1) * (2 * n_eff + 1) / 24.0 - tie_term))
    z = (w_plus - mu) / sigma
    log10p = log10_p_from_normal(z)
    out = {
        "label": label,
        "n_total": int(y.size),
        "n_zero_dropped": n_zero,
        "n_effective": int(n_eff),
        "statistic": float(res.statistic),
        "w_plus": w_plus,
        "z_normal_approx": float(z),
        "pvalue_raw": float(res.pvalue),
        "pvalue_underflowed": bool(res.pvalue == 0.0),
        "log10_pvalue": log10p,
        "alternative": "greater",
        "zero_method": "wilcox",
    }
    disp = (
        f"p < 1e-300 (underflow; log10(p) = {log10p:.1f})"
        if out["pvalue_underflowed"]
        else f"p = {out['pvalue_raw']:.6g}"
    )
    out["pvalue_reported"] = disp
    print(f"[{label}] W={out['statistic']:.6g}  zeros dropped={n_zero}  n_eff={n_eff:,}")
    print(f"[{label}] z={z:.3f}  {disp}")
    return out


t0 = time.monotonic()
wil_primary = wilcoxon_report(Y, "PRIMARY_FINAL_COHORT")
wil_sec = round(time.monotonic() - t0, 2)
print(f"\nwilcoxon runtime {wil_sec}s")

[PRIMARY_FINAL_COHORT] W=6.40555e+12  zeros dropped=196718  n_eff=3,639,270
[PRIMARY_FINAL_COHORT] z=1544.070  p < 1e-300 (underflow; log10(p) = -517715.4)

wilcoxon runtime 0.6s


## 05 — Sign test (mandatory robustness)

Ties (`Y == 0`) are excluded from the binomial denominator. The tie count and share are reported
separately, as the protocol requires.

In [5]:
def sign_report(y: np.ndarray, label: str) -> dict:
    pos = int((y > 0.0).sum())
    neg = int((y < 0.0).sum())
    ties = int((y == 0.0).sum())
    n_eff = pos + neg
    res = st.binomtest(pos, n=n_eff, p=0.5, alternative="greater")
    # underflow-safe log10(p) via the normal approximation to Binomial(n_eff, 0.5)
    z = (pos - n_eff / 2.0) / np.sqrt(n_eff / 4.0)
    log10p = log10_p_from_normal(z)
    out = {
        "label": label,
        "positive": pos,
        "negative": neg,
        "ties": ties,
        "tie_share": ties / y.size,
        "n_effective": n_eff,
        "pvalue_raw": float(res.pvalue),
        "pvalue_underflowed": bool(res.pvalue == 0.0),
        "log10_pvalue": log10p,
        "z_normal_approx": float(z),
        "alternative": "greater",
    }
    disp = (
        f"p < 1e-300 (underflow; log10(p) = {log10p:.1f})"
        if out["pvalue_underflowed"]
        else f"p = {out['pvalue_raw']:.6g}"
    )
    out["pvalue_reported"] = disp
    print(f"[{label}] +{pos:,}  -{neg:,}  ties={ties:,} ({out['tie_share']:.6%})  n_eff={n_eff:,}")
    print(f"[{label}] {disp}")
    return out


sign_primary = sign_report(Y, "PRIMARY_FINAL_COHORT")

[PRIMARY_FINAL_COHORT] +3,375,095  -264,175  ties=196,718 (5.128223%)  n_eff=3,639,270
[PRIMARY_FINAL_COHORT] p < 1e-300 (underflow; log10(p) = -577458.1)


## 06 — Bootstrap equivalence benchmark

The full run uses a monotone-indexing shortcut: because the outcome is sorted once, the median of a
resample equals the sorted array evaluated at the middle order statistics of the drawn indices. This
cell verifies that shortcut against a direct `np.median` reference on the same drawn indices. Both
must agree bit-for-bit; any mismatch invalidates the CI regardless of runtime.

In [6]:
Y_SORTED = np.sort(Y)
N = Y_SORTED.size
LO, HI = (N // 2) - 1, N // 2  # N is even; median averages these two order statistics
assert N % 2 == 0, N


def boot_medians_fast(y_sorted: np.ndarray, b: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n = y_sorted.size
    out = np.empty(b, dtype=np.float64)
    for i in range(b):
        idx = rng.integers(0, n, size=n)
        part = np.partition(idx, (LO, HI))
        out[i] = 0.5 * (y_sorted[part[LO]] + y_sorted[part[HI]])
    return out


def boot_medians_reference(y_sorted: np.ndarray, b: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n = y_sorted.size
    out = np.empty(b, dtype=np.float64)
    for i in range(b):
        idx = rng.integers(0, n, size=n)
        out[i] = float(np.median(y_sorted[idx]))
    return out


BENCH_B = 25
t0 = time.monotonic()
bench_fast = boot_medians_fast(Y_SORTED, BENCH_B, BOOTSTRAP_SEED)
bench_ref = boot_medians_reference(Y_SORTED, BENCH_B, BOOTSTRAP_SEED)
bench_sec = round(time.monotonic() - t0, 2)

mismatch = int((bench_fast != bench_ref).sum())
equivalence = {
    "benchmark_replicates": BENCH_B,
    "seed": BOOTSTRAP_SEED,
    "mismatch_replicates": mismatch,
    "max_abs_difference": float(np.max(np.abs(bench_fast - bench_ref))),
    "status": "PASS" if mismatch == 0 else "FAIL",
    "runtime_sec": bench_sec,
    "note": "monotone-indexing shortcut vs direct np.median on identical drawn indices",
}
print(f"replicates={BENCH_B}  mismatch={mismatch}  max|diff|={equivalence['max_abs_difference']}")
print(f"BOOTSTRAP_EQUIVALENCE_STATUS = {equivalence['status']}")
if mismatch != 0:
    raise SystemExit("BOOTSTRAP_EQUIVALENCE_FAILED")

replicates=25  mismatch=0  max|diff|=0.0
BOOTSTRAP_EQUIVALENCE_STATUS = PASS


## 07 — Full primary bootstrap CI

`B = 2000`, pair-level iid resampling, seed `969634713`, percentile quantiles 0.025 / 0.975.

In [7]:
t0 = time.monotonic()
boot_primary = boot_medians_fast(Y_SORTED, BOOTSTRAP_B, BOOTSTRAP_SEED)
boot_sec = round(time.monotonic() - t0, 2)

median_primary = float(np.median(Y))
ci_primary = [float(np.quantile(boot_primary, CI_Q[0])), float(np.quantile(boot_primary, CI_Q[1]))]
exp_median_primary = float(np.exp(median_primary))

print(f"B={BOOTSTRAP_B}  seed={BOOTSTRAP_SEED}  runtime {boot_sec}s")
print(f"median(logTP)      {median_primary:.10f}")
print(f"95% percentile CI  [{ci_primary[0]:.10f}, {ci_primary[1]:.10f}]")
print(f"exp(median)        {exp_median_primary:.10f}")
print(f"bootstrap replicate sd {float(boot_primary.std(ddof=1)):.3e}")

B=2000  seed=969634713  runtime 57.75s
median(logTP)      0.2876820725
95% percentile CI  [0.2876820725, 0.2876820725]
exp(median)        1.3333333333
bootstrap replicate sd 5.553e-17


## 08 — Known-direction sensitivity

`translation_direction != 'UNKNOWN'`. Same estimand, tests, bootstrap settings and seed. This is
reported beside the primary result and does not replace it.

In [8]:
Y_KNOWN = Y[KNOWN]
print(f"known-direction N = {Y_KNOWN.size:,}")

wil_known = wilcoxon_report(Y_KNOWN, "KNOWN_DIRECTION_ONLY")
sign_known = sign_report(Y_KNOWN, "KNOWN_DIRECTION_ONLY")

Y_KNOWN_SORTED = np.sort(Y_KNOWN)
NK = Y_KNOWN_SORTED.size
LOK, HIK = (NK - 1) // 2, NK // 2  # handles odd or even NK


def boot_medians_fast_k(y_sorted: np.ndarray, b: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    n = y_sorted.size
    out = np.empty(b, dtype=np.float64)
    for i in range(b):
        idx = rng.integers(0, n, size=n)
        part = np.partition(idx, (LOK, HIK))
        out[i] = 0.5 * (y_sorted[part[LOK]] + y_sorted[part[HIK]])
    return out


t0 = time.monotonic()
boot_known = boot_medians_fast_k(Y_KNOWN_SORTED, BOOTSTRAP_B, BOOTSTRAP_SEED)
boot_known_sec = round(time.monotonic() - t0, 2)

median_known = float(np.median(Y_KNOWN))
ci_known = [float(np.quantile(boot_known, CI_Q[0])), float(np.quantile(boot_known, CI_Q[1]))]
exp_median_known = float(np.exp(median_known))
share_gt1_known = float((Y_KNOWN > 0).sum() / Y_KNOWN.size)

print(f"median(logTP)      {median_known:.10f}")
print(f"95% percentile CI  [{ci_known[0]:.10f}, {ci_known[1]:.10f}]")
print(f"exp(median)        {exp_median_known:.10f}")

known-direction N = 3,785,441


[KNOWN_DIRECTION_ONLY] W=6.23753e+12  zeros dropped=194084  n_eff=3,591,357
[KNOWN_DIRECTION_ONLY] z=1533.636  p < 1e-300 (underflow; log10(p) = -510742.4)
[KNOWN_DIRECTION_ONLY] +3,330,539  -260,818  ties=194,084 (5.127117%)  n_eff=3,591,357
[KNOWN_DIRECTION_ONLY] p < 1e-300 (underflow; log10(p) = -569765.7)


median(logTP)      0.2876820725
95% percentile CI  [0.2876820725, 0.2876820725]
exp(median)        1.3333333333


## 09 — Primary results table

In [9]:
rows = [
    {
        "cohort": "PRIMARY_FINAL_COHORT",
        "N": int(Y.size),
        "median_logTP": median_primary,
        "bootstrap_95CI_low": ci_primary[0],
        "bootstrap_95CI_high": ci_primary[1],
        "exp_median_logTP": exp_median_primary,
        "wilcoxon_W": wil_primary["statistic"],
        "wilcoxon_p": wil_primary["pvalue_reported"],
        "sign_positive": sign_primary["positive"],
        "sign_negative": sign_primary["negative"],
        "sign_ties": sign_primary["ties"],
        "sign_p": sign_primary["pvalue_reported"],
        "share_TP_gt_1": desc["share_TP_gt_1"],
    },
    {
        "cohort": "KNOWN_DIRECTION_ONLY",
        "N": int(Y_KNOWN.size),
        "median_logTP": median_known,
        "bootstrap_95CI_low": ci_known[0],
        "bootstrap_95CI_high": ci_known[1],
        "exp_median_logTP": exp_median_known,
        "wilcoxon_W": wil_known["statistic"],
        "wilcoxon_p": wil_known["pvalue_reported"],
        "sign_positive": sign_known["positive"],
        "sign_negative": sign_known["negative"],
        "sign_ties": sign_known["ties"],
        "sign_p": sign_known["pvalue_reported"],
        "share_TP_gt_1": share_gt1_known,
    },
]

hdr = ["cohort", "N", "median_logTP", "bootstrap_95CI_low", "bootstrap_95CI_high",
       "exp_median_logTP", "wilcoxon_W", "wilcoxon_p", "sign_positive", "sign_negative",
       "sign_ties", "sign_p", "share_TP_gt_1"]
print("| " + " | ".join(hdr) + " |")
print("|" + "|".join(["---"] * len(hdr)) + "|")
for r in rows:
    cells = []
    for h in hdr:
        v = r[h]
        cells.append(f"{v:,}" if isinstance(v, int) else (f"{v:.10g}" if isinstance(v, float) else str(v)))
    print("| " + " | ".join(cells) + " |")

print("\nshare_TP_gt_1 is a descriptive contextual statistic. It is NOT the sign test and does")
print("not carry the sign test's inferential role.")

| cohort | N | median_logTP | bootstrap_95CI_low | bootstrap_95CI_high | exp_median_logTP | wilcoxon_W | wilcoxon_p | sign_positive | sign_negative | sign_ties | sign_p | share_TP_gt_1 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| PRIMARY_FINAL_COHORT | 3,835,988 | 0.2876820725 | 0.2876820725 | 0.2876820725 | 1.333333333 | 6.405551963e+12 | p < 1e-300 (underflow; log10(p) = -517715.4) | 3,375,095 | 264,175 | 196,718 | p < 1e-300 (underflow; log10(p) = -577458.1) | 0.8798502498 |
| KNOWN_DIRECTION_ONLY | 3,785,441 | 0.2876820725 | 0.2876820725 | 0.2876820725 | 1.333333333 | 6.237534312e+12 | p < 1e-300 (underflow; log10(p) = -510742.4) | 3,330,539 | 260,818 | 194,084 | p < 1e-300 (underflow; log10(p) = -569765.7) | 0.8798285325 |

share_TP_gt_1 is a descriptive contextual statistic. It is NOT the sign test and does
not carry the sign test's inferential role.


## 10 — Result artifact and claim boundary

In [10]:
results = {
    "artifact_id": "NB08_RQ1_RESULTS_v001",
    "decision_id": "RD-RQ1-FIRST-RESULT-01",
    "protocol_id": "NB08_RQ1_PROTOCOL_v001",
    "change_requests": ["CR-RQ1-BOOTSTRAP-FAST-2000-01"],
    "git": {
        "base_main_sha": BASE_MAIN_SHA,
        "rq1_decision_sha": RQ1_DECISION_SHA,
        "rq1_cohort_sha": RQ1_COHORT_SHA,
        "rq1_protocol_sha": RQ1_PROTOCOL_SHA,
        "branch": "research/nb08-rq1-primary-20260817",
    },
    "source": {
        "artifact_path": str(D04.relative_to(ROOT)),
        "artifact_sha256": actual_sha,
        "pair_set_hash": pair_set_hash,
        "row_count": int(n_rows),
        "distinct_pair_id": int(n_distinct),
        "eda_v2_used_as_inference_source": False,
    },
    "software": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "duckdb": duckdb.__version__,
        "pyarrow": pyarrow.__version__,
    },
    "outcome": "log_token_premium",
    "estimand": "Median(log_token_premium)",
    "hypothesis": {"H0": "theta = 0", "H1": "theta > 0", "alternative": "greater"},
    "descriptive": desc,
    "primary": {
        "cohort": "PRIMARY_FINAL_COHORT",
        "n": int(Y.size),
        "median_logTP": median_primary,
        "exp_median_logTP": exp_median_primary,
        "bootstrap_ci95": ci_primary,
        "wilcoxon": wil_primary,
        "sign_test": sign_primary,
        "share_TP_gt_1": desc["share_TP_gt_1"],
    },
    "sensitivity_known_direction": {
        "cohort": "KNOWN_DIRECTION_ONLY",
        "n": int(Y_KNOWN.size),
        "median_logTP": median_known,
        "exp_median_logTP": exp_median_known,
        "bootstrap_ci95": ci_known,
        "wilcoxon": wil_known,
        "sign_test": sign_known,
        "share_TP_gt_1": share_gt1_known,
    },
    "bootstrap": {
        "method": "percentile, pair_id row-level iid resampling",
        "implementation": "monotone-indexing order-statistic shortcut on the sorted outcome",
        "B": BOOTSTRAP_B,
        "seed": BOOTSTRAP_SEED,
        "seed_source_string": BOOTSTRAP_SEED_SOURCE,
        "seed_sha256": hashlib.sha256(BOOTSTRAP_SEED_SOURCE.encode()).hexdigest(),
        "quantiles": list(CI_Q),
        "equivalence_audit": equivalence,
        "runtime_sec_primary": boot_sec,
        "runtime_sec_known_direction": boot_known_sec,
        "resampling_unit_caveat": (
            "Row-level iid bootstrap does not account for source dependence or source imbalance. "
            "Source-stratified and dependence-sensitive robustness is deferred to an NB08 appendix "
            "or NB11 and is not a prerequisite for this first-result release."
        ),
    },
    "runtime_sec": {"load": load_sec, "wilcoxon": wil_sec},
    "policies": {
        "zero_method": "wilcox",
        "sign_test_ties": "excluded from the binomial denominator",
        "nonfinite_policy": "hard fail, no post-hoc deletion",
        "pvalue_policy": "p = 0 is never reported; log10(p) given on underflow",
        "primary_exclusion_rule": "NONE beyond the frozen final cohort",
    },
    "claim_status": {
        "permitted": (
            "Under the fixed o200k_base raw-text Track A measurement and the defined final paired "
            "KO-EN cohort, statistical evidence was observed that the pair-level median log token "
            "premium is greater than zero."
        ),
        "prohibited": [
            "Korean is intrinsically inefficient for AI",
            "generalization to all tokenizers",
            "morphology is the cause",
            "any domain effect claim",
            "reasoning degradation",
            "any fixed API cost increase figure claimed unconditionally",
            "any causal language",
        ],
        "notes": [
            "exp(median(logTP)) is a median-scale quantity and is NOT the aggregate token ratio.",
            "share_TP_gt_1 is descriptive context, not the sign test.",
            "Wilcoxon signed-rank carries a symmetry/location-shift interpretation; the sign test "
            "is the distribution-free robustness check on the median estimand.",
        ],
    },
    "created_at_kst": dt.datetime.now(KST).isoformat(timespec="seconds"),
}

out_path = ROOT / "ssot_nb01/04_NB08_RQ1_RESULTS_v001.json"
out_path.write_text(json.dumps(results, ensure_ascii=False, indent=2, sort_keys=True) + "\n")
con.close()
print(f"written {out_path.relative_to(ROOT)}")
print("\nRQ1_PRIMARY_INFERENCE_COMPLETE")

written ssot_nb01/04_NB08_RQ1_RESULTS_v001.json

RQ1_PRIMARY_INFERENCE_COMPLETE


---

# POST-PROTOCOL SSOT CLOSEOUT

**Authority**: `KOEN-TP-RS-001` · `RD-SSOT-CANONICAL-RETURN-01`
**Closeout ID**: `NB08_RQ1_SSOT_CLOSEOUT_v001`

위 열 개 cell은 frozen protocol `NB08_RQ1_PROTOCOL_v001`의 실행이며 evidence-of-record다.
이 섹션은 그 결과를 **바꾸지 않고**, SSOT가 요구하되 first release에서 기술되지 않았던
방법상의 사항만 닫는다.

RQ1의 primary result는 이미 산출되었다. 남은 것은 결과가 아니라 **방법의 기술(記述)**이다.
어떤 검정이 실제로 무엇을 추론했는지, 보고된 수치가 어떤 종류의 양인지, SSOT가 요구했으나
first release가 생략한 절차가 무엇인지를 정리하는 일이다. 이 다섯 항목이 해소되지 않으면
NB08은 "결과가 있다"고는 말할 수 있어도 "닫혔다"고는 말할 수 없다.

닫는 항목:

| | 항목 | 성격 |
|---|---|---|
| 2.1 | ties-excluded sign test를 `CONDITIONAL_NONZERO_SIGN_TEST`로 재분류 | 라벨 정정, 수치 불변 |
| 2.2 | `TIE_AWARE_MEDIAN_SIGN_ROBUSTNESS` 추가 (전체 `N`을 denominator로) | 신규 robustness |
| 2.3 | `SOURCE_STRATIFIED_BOOTSTRAP_SENSITIVITY` (SSOT §17.2) | 신규 sensitivity |
| 2.4 | `p < 1e-300` 보고 형식과 `APPROX_LOG10_P_NORMAL_DIAGNOSTIC` 라벨 | 보고 규약 |
| 2.5 | CI degeneracy 서술의 보장 표현 교정 | 문서 교정 |

**변경하지 않는 대상** — estimand, cohort, primary Wilcoxon signed-rank,
first-release pair-level bootstrap(`B = 2000`, seed `969634713`), 그리고 기존 결과 수치.
어느 하나라도 재계산 값이 달라지면 이 섹션은 HARD STOP한다.

## C0 — HARD STOP guard

닫기 작업이 대상을 바꾸지 않았음을 먼저 증명한다. 아래 값은 evidence-of-record commit
`502bc128f6b5855f1648802cc990b715808f26f3`에서 고정된 것이며, 위 protocol cell이 방금
산출한 살아 있는 변수와 대조한다. 하나라도 어긋나면 계산을 시작하지 않는다.

In [11]:
import numpy as np

CLOSEOUT_ID = "NB08_RQ1_SSOT_CLOSEOUT_v001"
PRIMARY_RESULT_SHA = "502bc128f6b5855f1648802cc990b715808f26f3"

# evidence-of-record 고정값 (덮어쓰지 않고 대조에만 사용한다)
FROZEN = {
    "primary_median_logTP": 0.28768207245178085,
    "primary_ci95": [0.28768207245178085, 0.28768207245178085],
    "primary_exp_median": 1.3333333333333333,
    "primary_wilcoxon_W": 6405551963244.0,
    "primary_sign_pos": 3375095, "primary_sign_neg": 264175, "primary_sign_ties": 196718,
    "primary_n": 3835988,
    "known_n": 3785441,
    "known_median_logTP": 0.28768207245178085,
    "known_wilcoxon_W": 6237534311943.0,
    "known_sign_pos": 3330539, "known_sign_neg": 260818, "known_sign_ties": 194084,
}

guard = {
    "primary median": (median_primary, FROZEN["primary_median_logTP"]),
    "primary CI": (ci_primary, FROZEN["primary_ci95"]),
    "primary exp(median)": (exp_median_primary, FROZEN["primary_exp_median"]),
    "primary Wilcoxon W": (wil_primary["statistic"], FROZEN["primary_wilcoxon_W"]),
    "primary sign +": (sign_primary["positive"], FROZEN["primary_sign_pos"]),
    "primary sign -": (sign_primary["negative"], FROZEN["primary_sign_neg"]),
    "primary sign ties": (sign_primary["ties"], FROZEN["primary_sign_ties"]),
    "primary N": (int(Y.size), FROZEN["primary_n"]),
    "known N": (int(Y_KNOWN.size), FROZEN["known_n"]),
    "known median": (median_known, FROZEN["known_median_logTP"]),
    "known Wilcoxon W": (wil_known["statistic"], FROZEN["known_wilcoxon_W"]),
    "known sign +": (sign_known["positive"], FROZEN["known_sign_pos"]),
    "bootstrap B": (BOOTSTRAP_B, 2000),
    "bootstrap seed": (BOOTSTRAP_SEED, 969634713),
    "D-04 sha256": (actual_sha, D04_SHA256),
}
for name, (got, want) in guard.items():
    print(f"  {name:22s} {'OK  ' if got == want else 'FAIL'} {str(got)[:40]}")
if any(g != w for g, w in guard.values()):
    raise SystemExit("PRIMARY_RECOMPUTATION_DIVERGED — HARD STOP")
print("\nPRIMARY_PROTOCOL_UNCHANGED")

  primary median         OK   0.28768207245178085
  primary CI             OK   [0.28768207245178085, 0.2876820724517808
  primary exp(median)    OK   1.3333333333333333
  primary Wilcoxon W     OK   6405551963244.0
  primary sign +         OK   3375095
  primary sign -         OK   264175
  primary sign ties      OK   196718
  primary N              OK   3835988
  known N                OK   3785441
  known median           OK   0.28768207245178085
  known Wilcoxon W       OK   6237534311943.0
  known sign +           OK   3330539
  bootstrap B            OK   2000
  bootstrap seed         OK   969634713
  D-04 sha256            OK   1c30e3276222dd94885ae4f79fc6ab5c45e4e262

PRIMARY_PROTOCOL_UNCHANGED


## C1 — 2.1 · `CONDITIONAL_NONZERO_SIGN_TEST` 재분류

first release가 "sign test"로 보고한 검정은 `logTP = 0`인 pair를 denominator에서 제외했다.
이는 계산 착오가 아니라 표준적인 zero-exclusion 관례다. 그러나 그 결과 검정이 추론하는
대상이 달라진다. 제외된 검정이 다루는 모수는 주변확률 `P(Y > 0)`이 아니라 조건부 확률이다.

$$H_0:\ P(Y>0 \mid Y \neq 0) = \tfrac12
\qquad H_1:\ P(Y>0 \mid Y \neq 0) > \tfrac12$$

즉 "Korean이 더 많은 token을 쓰는가"가 아니라 **"두 언어의 token 수가 다르다는 조건 아래"**
Korean 쪽이 더 많은가를 물은 것이다. 이 cohort에서 tie는 196,718건, 전체의 5.13%로 무시할 수
있는 규모가 아니다. 조건부 모수와 주변 모수가 실질적으로 갈라질 만큼의 질량이므로, 명칭을
정확히 하는 일은 형식적 손질이 아니라 추론 대상의 정정이다.

검정 자체는 삭제하지 않는다. 그것은 non-zero observation 내부의 polarity robustness로서
유효하다. 다만 **tie가 존재하는 population median에 대한 distribution-free exact statement로
부르지 않는다.** 수치는 그대로이고, 변하는 것은 그 수치가 무엇에 대한 진술인지에 대한
기술이다.

In [12]:
CONDITIONAL_NONZERO_SIGN_TEST = {
    "label": "CONDITIONAL_NONZERO_SIGN_TEST",
    "former_label": "sign test (ties excluded)",
    "estimand": "P(Y > 0 | Y != 0)",
    "null": "P(Y > 0 | Y != 0) = 0.5",
    "alternative": "greater",
    "role": "non-zero observation 내부의 polarity robustness",
    "not_a": "tie가 존재하는 population median에 대한 distribution-free exact statement",
    "primary": {
        "positive": sign_primary["positive"], "negative": sign_primary["negative"],
        "ties_excluded": sign_primary["ties"], "n_effective": sign_primary["n_effective"],
        "point_estimate": sign_primary["positive"] / sign_primary["n_effective"],
        "pvalue_reported": "p < 1e-300",
        "APPROX_LOG10_P_NORMAL_DIAGNOSTIC": sign_primary["log10_pvalue"],
        "z_normal_approx": sign_primary["z_normal_approx"],
    },
    "known_direction": {
        "positive": sign_known["positive"], "negative": sign_known["negative"],
        "ties_excluded": sign_known["ties"], "n_effective": sign_known["n_effective"],
        "point_estimate": sign_known["positive"] / sign_known["n_effective"],
        "pvalue_reported": "p < 1e-300",
        "APPROX_LOG10_P_NORMAL_DIAGNOSTIC": sign_known["log10_pvalue"],
        "z_normal_approx": sign_known["z_normal_approx"],
    },
}
for k in ("primary", "known_direction"):
    d = CONDITIONAL_NONZERO_SIGN_TEST[k]
    print(f"[{k}] P(Y>0 | Y!=0) = {d['point_estimate']:.6f}   n_eff = {d['n_effective']:,}   "
          f"tie {d['ties_excluded']:,} 제외")
print("\n수치는 first release와 동일하다. 재분류는 라벨과 해석 범위에만 적용된다.")

[primary] P(Y>0 | Y!=0) = 0.927410   n_eff = 3,639,270   tie 196,718 제외
[known_direction] P(Y>0 | Y!=0) = 0.927376   n_eff = 3,591,357   tie 194,084 제외

수치는 first release와 동일하다. 재분류는 라벨과 해석 범위에만 적용된다.


## C2 — 2.2 · `TIE_AWARE_MEDIAN_SIGN_ROBUSTNESS`

조건부 검정만으로는 `Median(logTP) > 0`이라는 **주변** 명제를 직접 지지할 수 없다.
중앙값에 대한 귀무가설은 tie를 포함한 전체 분포에 대한 진술이기 때문이다.

$$H_0:\ P(Y>0) \le \tfrac12 \qquad H_1:\ P(Y>0) > \tfrac12$$

`Median(Y) = 0`이면 `P(Y > 0)`는 1/2을 넘을 수 없으나, tie가 질량의 일부를 가져가므로
1/2보다 작을 수 있다. 등호가 아니라 부등호가 되는 이 구조 때문에 귀무가설은 합성가설이며,
검정은 **least-favourable null** 인 `Binomial(N, 0.5)`의 upper tail로 수행한다. `K`는
`logTP > 0`인 행의 수이고 `N`은 tie를 포함한 전체 primary row 수다. ties를 denominator에서
버리지 않는다.

이 검정은 조건부 검정보다 반드시 약한 증거를 준다. 그럼에도 추가하는 이유는 두 가지다.
첫째, 이것이 median에 대한 주장과 정확히 같은 대상을 검정한다. 둘째, tie를 버리는 절차가
결론을 만들어낸 것이 아님을 보인다. tie 196,718건을 전부 대립가설에 불리한 쪽에 세워도
결론이 유지되는지가 관건이며, 유지된다면 zero-exclusion은 결론의 원인이 아니라 관례에
불과했다는 뜻이 된다. known-direction cohort에도 동일하게 계산한다.

In [13]:
import scipy.stats as st

def tie_aware_sign(y: np.ndarray, label: str) -> dict:
    K = int((y > 0.0).sum())
    n_neg = int((y < 0.0).sum())
    n_tie = int((y == 0.0).sum())
    n_all = int(y.size)                      # ties를 denominator에 포함한다
    res = st.binomtest(K, n=n_all, p=0.5, alternative="greater")
    z = (K - n_all / 2.0) / np.sqrt(n_all / 4.0)
    out = {
        "label": "TIE_AWARE_MEDIAN_SIGN_ROBUSTNESS",
        "cohort": label,
        "null": "P(Y > 0) <= 0.5   (least-favourable: Binomial(N, 0.5))",
        "alternative": "greater",
        "K_positive": K, "negative": n_neg, "ties_retained": n_tie,
        "N_denominator": n_all,
        "point_estimate": K / n_all,
        "excess_over_half": K / n_all - 0.5,
        "pvalue_raw": float(res.pvalue),
        "pvalue_underflowed": bool(res.pvalue == 0.0),
        "pvalue_reported": "p < 1e-300" if res.pvalue == 0.0 else f"p = {res.pvalue:.6g}",
        "APPROX_LOG10_P_NORMAL_DIAGNOSTIC": float(st.norm.logsf(z) / np.log(10.0)),
        "z_normal_approx": float(z),
        "direction_same_as_primary": (K / n_all) > 0.5,
    }
    print(f"[{label}] K={K:,} / N={n_all:,}  ->  P(Y>0) = {out['point_estimate']:.6f}")
    print(f"           tie {n_tie:,}건을 denominator에 유지  ·  z = {z:,.1f}  ·  {out['pvalue_reported']}")
    print(f"           1/2 초과폭 = {out['excess_over_half']:.6f}")
    return out


TIE_AWARE = {
    "PRIMARY_FINAL_COHORT": tie_aware_sign(Y, "PRIMARY_FINAL_COHORT"),
    "KNOWN_DIRECTION_ONLY": tie_aware_sign(Y_KNOWN, "KNOWN_DIRECTION_ONLY"),
}
print()
c_hat = CONDITIONAL_NONZERO_SIGN_TEST["primary"]["point_estimate"]
t_hat = TIE_AWARE["PRIMARY_FINAL_COHORT"]["point_estimate"]
print(f"조건부 추정치 {c_hat:.6f}  ->  보수적 추정치 {t_hat:.6f}")
print("tie 전량을 반대편에 세워도 방향은 바뀌지 않는다. zero-exclusion은 결론의 원인이 아니다.")

[PRIMARY_FINAL_COHORT] K=3,375,095 / N=3,835,988  ->  P(Y>0) = 0.879850
           tie 196,718건을 denominator에 유지  ·  z = 1,487.9  ·  p < 1e-300
           1/2 초과폭 = 0.379850
[KNOWN_DIRECTION_ONLY] K=3,330,539 / N=3,785,441  ->  P(Y>0) = 0.879829
           tie 194,084건을 denominator에 유지  ·  z = 1,478.0  ·  p < 1e-300
           1/2 초과폭 = 0.379829

조건부 추정치 0.927410  ->  보수적 추정치 0.879850
tie 전량을 반대편에 세워도 방향은 바뀌지 않는다. zero-exclusion은 결론의 원인이 아니다.


## C3 — 2.3 · `SOURCE_STRATIFIED_BOOTSTRAP_SENSITIVITY`

SSOT §17.2는 *source 불균형이 큰 경우 source-stratified bootstrap을 기본으로 사용하고,
source cluster bootstrap은 source level 수가 충분할 때 보조로 수행한다*고 규정한다.

이 cohort의 실태를 먼저 본다. source level은 `logical_corpus` 기준 두 개이며 `source_id`와
전단사 관계다. 점유율은 025가 64.8%, 026이 35.2%로 약 1.84 : 1의 불균형이다. 그러나 더
중요한 것은 두 층의 **결과 분포가 실제로 다르다**는 점이다. 025의 median logTP는 0.2744,
026은 0.3087이며, `P(TP>1)`은 각각 83.4%와 96.5%로 13퍼센트포인트 넘게 벌어진다. 층 간
이질성이 이 정도라면 재표집에서 층 비율이 흔들릴 때 pooled median도 함께 흔들릴 수 있고,
이것이 §17.2가 stratified를 기본으로 두라고 한 이유다.

절차는 source composition을 고정한 채 각 stratum 내부에서 원래 stratum 크기만큼 복원추출한
뒤 층을 concatenate하여 overall `median(logTP)`를 계산하는 것이다. `B = 2000`.
sensitivity seed는 결과를 보기 전에 결정론적으로 도출했다.

```
seed = uint32( first_8_hex( SHA256("NB08_RQ1_SSOT_CLOSEOUT_v001|SOURCE_STRATIFIED") ) )
SHA256 = aa49bab84f2aa5899c72777b35360d16e4cc0d5c0e508e5fcfac2af4367f0089
seed   = 2856958648
```

**source cluster bootstrap은 수행하지 않는다.** level 수가 2에 불과해 cluster 재표집의
표집분포가 사실상 정의되지 않으며, 이는 §17.2가 명시한 *"source level 수가 충분할 때"* 조건에
미달한다. 생략이 아니라 조건 불충족에 따른 배제로 기록한다.

first release의 pair-level bootstrap은 그대로 유지되며 이 sensitivity가 그것을 대체하지
않는다.

In [14]:
import duckdb, hashlib, time
from tqdm.auto import tqdm

# 장시간 루프는 10초 간격으로 파일에 진행률을 남긴다(nbconvert는 출력을 버퍼링하므로
# 노트북 출력만으로는 실행 중 상태를 볼 수 없다).
PROGRESS_DIR = ROOT / ".runtime/nb08-closeout"
PROGRESS_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_LOG = PROGRESS_DIR / "progress.log"
_plog = PROGRESS_LOG.open("w", buffering=1)


def tq(iterable, desc, total):
    return tqdm(iterable, desc=desc, total=total, file=_plog,
                mininterval=10.0, miniters=1, ascii=True, ncols=90)

SENS_SEED_SOURCE = "NB08_RQ1_SSOT_CLOSEOUT_v001|SOURCE_STRATIFIED"
SENS_SEED = int(hashlib.sha256(SENS_SEED_SOURCE.encode()).hexdigest()[:8], 16)
assert SENS_SEED == 2856958648, SENS_SEED

# 결과와 stratum 라벨은 반드시 '하나의 질의'에서 함께 읽고 명시적으로 정렬한다.
# DuckDB는 별개 질의 간 행 순서를 보장하지 않으므로(병렬 스캔 + preserve_insertion_order=false),
# 두 번에 나눠 읽으면 라벨이 값과 어긋나 stratum이 무작위 분할이 되어 버린다.
con2 = duckdb.connect()
con2.execute("SET memory_limit='5GB'"); con2.execute("SET threads=8")
con2.execute("SET preserve_insertion_order=false")
joined = con2.execute(
    f"""SELECT t.log_token_premium AS y, p.logical_corpus AS src
        FROM read_parquet('{D04.as_posix()}') t
        JOIN read_parquet('{(ROOT / 'data/registry/PAIR_REGISTRY_v002.parquet').as_posix()}') p
          USING (pair_id)
        ORDER BY t.pair_id"""
).fetch_arrow_table()
Y_C = joined.column("y").to_numpy(zero_copy_only=False).astype(np.float64)
SRC = np.asarray(joined.column("src").to_pylist())
del joined
con2.close()

# 정합 guard — 같은 multiset이고 pooled 통계가 protocol과 일치해야 한다.
ALIGNMENT = {
    "loaded_in_single_query": True,
    "explicit_order_by": "t.pair_id",
    "same_size": bool(Y_C.size == Y.size),
    "same_multiset": bool(np.array_equal(np.sort(Y_C), np.sort(Y))),
    "pooled_median_matches_protocol": bool(float(np.median(Y_C)) == median_primary),
}
for k, v in ALIGNMENT.items():
    print(f"  alignment · {k:32s} {v}")
if not all(v for k, v in ALIGNMENT.items() if isinstance(v, bool)):
    raise SystemExit("SOURCE_LABEL_ALIGNMENT_FAILED — stratified bootstrap 중단")

order = np.argsort(Y_C, kind="stable")
Y_S = Y_C[order]
SRC_S = SRC[order]
NN = Y_S.size
LO2, HI2 = (NN // 2) - 1, NN // 2          # NN은 짝수
levels, counts = np.unique(SRC, return_counts=True)

strata = []
for lv in levels:
    pos = np.flatnonzero(SRC_S == lv)
    strata.append((str(lv), pos, pos.size))
    sub = Y_C[SRC == lv]
    print(f"  stratum {lv}: n={pos.size:,} ({pos.size/NN:.2%})  median={np.median(sub):.7f}  "
          f"P(TP>1)={float((sub>0).mean()):.5f}  tie={float((sub==0).mean()):.4%}")
print(f"\n  불균형 {max(counts)/min(counts):.3f} : 1   source level 수 = {len(levels)}")


def stratified_median_boot(b: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    out = np.empty(b, dtype=np.float64)
    buf = np.empty(NN, dtype=np.int64)
    for i in tq(range(b), "source-stratified bootstrap", b):
        off = 0
        for _lv, pos, m in strata:
            buf[off:off + m] = pos[rng.integers(0, m, size=m)]
            off += m
        part = np.partition(buf, (LO2, HI2))
        out[i] = 0.5 * (Y_S[part[LO2]] + Y_S[part[HI2]])
    return out


t0 = time.monotonic()
BOOT_STRAT = stratified_median_boot(BOOTSTRAP_B, SENS_SEED)
strat_sec = round(time.monotonic() - t0, 1)
_plog.write(f"DONE stratified bootstrap {strat_sec}s\n"); _plog.flush()
ci_strat = [float(np.quantile(BOOT_STRAT, CI_Q[0])), float(np.quantile(BOOT_STRAT, CI_Q[1]))]

SOURCE_STRATIFIED = {
    "label": "SOURCE_STRATIFIED_BOOTSTRAP_SENSITIVITY",
    "ssot_ref": "§17.2",
    "design": "source composition 고정, 각 stratum 내부에서 stratum 크기만큼 복원추출 후 concatenate",
    "statistic": "overall median(log_token_premium)",
    "B": BOOTSTRAP_B, "seed": SENS_SEED, "seed_source_string": SENS_SEED_SOURCE,
    "seed_sha256": hashlib.sha256(SENS_SEED_SOURCE.encode()).hexdigest(),
    "quantiles": list(CI_Q),
    "source_levels": [s[0] for s in strata],
    "source_shares": {s[0]: float(s[2] / NN) for s in strata},
    "imbalance_ratio": float(max(counts) / min(counts)),
    "ci95": ci_strat,
    "replicate_sd": float(BOOT_STRAT.std(ddof=1)),
    "degenerate": ci_strat[0] == ci_strat[1],
    "primary_pair_level_ci95": ci_primary,
    "agrees_with_primary_ci": ci_strat == ci_primary,
    "materially_reverses_conclusion": bool(ci_strat[1] <= 0.0),
    "cluster_bootstrap": "NOT_PERFORMED — source level 수 2로 §17.2의 '충분할 때' 조건 미달",
    "runtime_sec": strat_sec,
    "row_alignment_guard": ALIGNMENT,
}
print(f"\nsource-stratified  B={BOOTSTRAP_B}  seed={SENS_SEED}  {strat_sec}s")
print(f"  stratified 95% CI = [{ci_strat[0]!r}, {ci_strat[1]!r}]")
print(f"  primary    95% CI = [{ci_primary[0]!r}, {ci_primary[1]!r}]")
print(f"  두 CI 동일 = {SOURCE_STRATIFIED['agrees_with_primary_ci']}   "
      f"결론 반전 = {SOURCE_STRATIFIED['materially_reverses_conclusion']}")

/tmp/ipykernel_240542/1655343329.py:32: DeprecationWarning: fetch_arrow_table() is deprecated, use to_arrow_table() instead.
  ).fetch_arrow_table()


  alignment · loaded_in_single_query           True
  alignment · explicit_order_by                t.pair_id
  alignment · same_size                        True
  alignment · same_multiset                    True
  alignment · pooled_median_matches_protocol   True


  stratum 025: n=2,485,963 (64.81%)  median=0.2744368  P(TP>1)=0.83362  tie=7.3361%
  stratum 026: n=1,350,025 (35.19%)  median=0.3087355  P(TP>1)=0.96497  tie=1.0626%

  불균형 1.841 : 1   source level 수 = 2


source-stratified bootstrap:   0%|                               | 0/2000 [00:00<?, ?it/s]


source-stratified  B=2000  seed=2856958648  102.7s
  stratified 95% CI = [0.28768207245178085, 0.28768207245178085]
  primary    95% CI = [0.28768207245178085, 0.28768207245178085]
  두 CI 동일 = True   결론 반전 = False


## C4 — 2.4 · p-value 보고 규약

first release는 `p < 1e-300 (underflow; log10(p) = -517715.4)` 형태로 보고했다. `p = 0`을
피한 점은 옳았으나, `log10(p)` 값이 **exact p-value의 로그**처럼 읽힐 여지를 남겼다.
그렇지 않다.

이 수치는 검정통계량의 정규근사 `z`로부터 `log10 Φ̄(z)`를 계산한 것이다. 두 겹의 근사가
개입한다. 첫째, 검정통계량 표집분포가 정규분포로 근사된다는 가정. 둘째, 그 근사가
`z ≈ 1544`라는 극단 tail에서도 유효하다는 가정. 첫째는 중심 근방에서 잘 성립하지만 둘째는
**어떤 유한 표본으로도 검증할 수 없다** — 관측 가능한 사건이 그 영역에 존재하지 않기 때문이다.

따라서 `log10(p) = -517,715`는 "p가 10의 마이너스 51만 승"이라는 측정이 아니라, 검정통계량이
귀무분포 중심에서 얼마나 떨어져 있는지를 로그 척도로 옮긴 진단 지표다. 실질 정보는 지수의
자릿수가 아니라 `z` 그 자체에 있다.

규약을 다음과 같이 고정한다. scipy가 산출한 raw underflow 사실 자체는 보존한다. 논문 및
대외 보고에서는 `p < 1e-300`으로 제한한다. 정규근사 로그값은 `APPROX_LOG10_P_NORMAL_DIAGNOSTIC`
라벨을 달고 `z`와 함께 보고하며, exact log p처럼 표현하지 않는다.

이 교정은 SSOT §17.3의 세 원칙과 직접 맞닿는다. §17.3은 *p-value 단독 보고 금지*,
*effect size + CI 우선*, *표본이 매우 큰 경우 "statistically significant"보다 실제 premium
크기를 해석*할 것을 요구한다. `N = 3,835,988`에서 유의성은 정보를 거의 담지 않는다. 어떤 미미한
편차도 유의해지기 때문이다. 결론을 지지하는 것은 지수가 아니라 효과 크기다.

In [15]:
REPORTING = {
    "raw_underflow_preserved": True,
    "publication_form": "p < 1e-300",
    "diagnostic_label": "APPROX_LOG10_P_NORMAL_DIAGNOSTIC",
    "diagnostic_derivation": "log10 of the standard-normal upper tail at the statistic's normal-approximation z",
    "approximations": [
        "검정통계량 표집분포의 정규근사",
        "그 근사가 |z| ~ 1.5e3 극단 tail에서 유효하다는 가정 (유한 표본으로 검증 불가)",
    ],
    "prohibited": "exact log p처럼 표현하는 것",
    "ssot_ref": "§17.3 — p-value 단독 보고 금지 / effect size + CI 우선 / 대표본에서는 premium 크기 해석",
    "z_diagnostics": {
        "wilcoxon_primary": wil_primary["z_normal_approx"],
        "wilcoxon_known": wil_known["z_normal_approx"],
        "conditional_nonzero_sign_primary": sign_primary["z_normal_approx"],
        "tie_aware_primary": TIE_AWARE["PRIMARY_FINAL_COHORT"]["z_normal_approx"],
        "tie_aware_known": TIE_AWARE["KNOWN_DIRECTION_ONLY"]["z_normal_approx"],
    },
}
print("정규근사 z (진단 지표) — 보고 형식은 모두 p < 1e-300")
for k, v in REPORTING["z_diagnostics"].items():
    print(f"  {k:34s} z = {v:>10,.1f}")
print("\n세 검정 모두 exact p는 double precision에서 underflow한다.")
print("보고되는 것은 p가 아니라 귀무분포 중심으로부터의 거리다.")

정규근사 z (진단 지표) — 보고 형식은 모두 p < 1e-300
  wilcoxon_primary                   z =    1,544.1
  wilcoxon_known                     z =    1,533.6
  conditional_nonzero_sign_primary   z =    1,630.7
  tie_aware_primary                  z =    1,487.9
  tie_aware_known                    z =    1,478.0

세 검정 모두 exact p는 double precision에서 underflow한다.
보고되는 것은 p가 아니라 귀무분포 중심으로부터의 거리다.


## C5 — 2.5 · CI degeneracy 서술의 교정

`ssot_nb01/05_NB08_RQ1_CI_DEGENERACY_NOTE.md`는 후속 항목으로 "non-degenerate interval을
얻으려면 격자를 존중하는 방법을 쓸 수 있다"고 적었다. 이 문장은 그런 방법이 **폭이 0이 아닌
구간을 보장한다**는 뜻으로 읽힌다. 그 보장은 성립하지 않는다.

중앙값에 대한 exact order-statistic interval은 정렬된 표본의 두 순서통계량을 그대로 끝점으로
취한다. 따라서 두 순서통계량이 같은 point mass 안에 들어가면 **그 구간 역시 퇴화한다**.
퇴화는 bootstrap이라는 절차의 성질이 아니라 자료가 격자 위에 놓여 있다는 사실의 성질이다.
검증 가능한 명제이므로 아래에서 직접 계산한다.

교정된 문구는 다음과 같다.

> Alternative lattice-aware or dependence-aware intervals may provide a different uncertainty
> characterization, but may still collapse at the same point mass.

관측된 primary degenerate CI 자체는 수정하지 않는다. 그것은 자료에 충실한 결과다.

In [16]:
zq = st.norm.ppf(1 - 0.025)
lo_rank = max(1, int(np.floor(NN / 2 - zq * np.sqrt(NN) / 2)))
hi_rank = min(NN, int(np.ceil(NN / 2 + zq * np.sqrt(NN) / 2)) + 1)
ci_order = [float(Y_S[lo_rank - 1]), float(Y_S[hi_rank - 1])]

med_val = float(np.median(Y))
mass = int((Y == med_val).sum())
below = int((Y < med_val).sum())
span = (below + 1, below + mass)

DEGENERACY = {
    "primary_observed_ci_unchanged": ci_primary,
    "corrected_wording": ("Alternative lattice-aware or dependence-aware intervals may provide a "
                          "different uncertainty characterization, but may still collapse at the "
                          "same point mass."),
    "removed_wording": "for a non-degenerate interval ... (alternative method가 non-degenerate CI를 보장한다는 함의)",
    "order_statistic_interval": {
        "method": "exact order-statistic interval for the median (normal-approximated ranks)",
        "rank_low": lo_rank, "rank_high": hi_rank, "ci95": ci_order,
        "degenerate": ci_order[0] == ci_order[1],
    },
    "point_mass_at_median": mass,
    "point_mass_rank_span": list(span),
    "both_ranks_inside_point_mass": bool(lo_rank >= span[0] and hi_rank <= span[1]),
    "source_stratified_also_degenerate": SOURCE_STRATIFIED["degenerate"],
}
print(f"order-statistic 95% CI  순위 {lo_rank:,} .. {hi_rank:,}")
print(f"  구간 = [{ci_order[0]!r}, {ci_order[1]!r}]   퇴화 = {DEGENERACY['order_statistic_interval']['degenerate']}")
print(f"  median point mass 순위 구간 = {span[0]:,} .. {span[1]:,}  (질량 {mass:,}행)")
print(f"  두 끝점이 모두 point mass 내부 = {DEGENERACY['both_ranks_inside_point_mass']}")
print(f"  source-stratified CI도 퇴화 = {DEGENERACY['source_stratified_also_degenerate']}")
print()
print("bootstrap과 무관한 exact 절차, 그리고 설계가 다른 stratified bootstrap이")
print("같은 결론에 도달한다. degeneracy는 방법의 산물이 아니라 자료의 성질이다.")

order-statistic 95% CI  순위 1,916,074 .. 1,919,915
  구간 = [0.28768207245178085, 0.28768207245178085]   퇴화 = True
  median point mass 순위 구간 = 1,841,885 .. 1,964,924  (질량 123,040행)
  두 끝점이 모두 point mass 내부 = True
  source-stratified CI도 퇴화 = True

bootstrap과 무관한 exact 절차, 그리고 설계가 다른 stratified bootstrap이
같은 결론에 도달한다. degeneracy는 방법의 산물이 아니라 자료의 성질이다.


## C6 — 해석 (SSOT §17.3)

§17.3은 표본이 매우 클 때 *"statistically significant"보다 실제 premium 크기를 해석*할 것을
요구한다. 아래는 그 요구에 따른 읽기다.

**효과의 크기.** 중앙 pair는 Korean 4 token 대 English 3 token, 곧 33.3%의 추가 token을
요구한다. 통계적 미세 편차가 아니라 실무적으로 즉시 체감되는 크기다. 동시에 median TP가
`4/3`이라는 작은 정수비에 정확히 놓였다는 사실은 우연이 아니라 격자 구조의 직접적 귀결이다.
중앙값 근방의 질량이 몇 개의 단순 비율에 집중되어 있으므로, 중앙 pair를 기술하는 가장 정직한
방식은 소수점 이하 여러 자리의 실수가 아니라 "4 대 3"이라는 비율 그 자체다.

**분포적 크기.** `P(TP > 1) = 87.99%`는 premium이 중앙값에만 국한된 현상이 아님을 말한다.
대다수 pair가 같은 방향을 가리키므로, 이 결과는 소수의 극단값이 중앙값을 끌어올린 결과가 아니다.
tie를 전량 반대편에 세운 보수적 검정에서도 방향이 유지된다는 사실(C2)이 이를 다시 확인한다.

**층 간 이질성.** 가장 해석에 주의가 필요한 부분이다. 두 source의 median과 `P(TP>1)`은 서로
뚜렷이 다르며, pooled median `ln(4/3)`은 두 층 어느 쪽의 median과도 일치하지 않는다. 이는
"한국어의 token premium이 얼마인가"라는 질문에 단일한 답이 없을 수 있음을 시사한다.
다만 이 관찰로부터 **source가 원인이라거나 domain 효과가 있다고 말해서는 안 된다.** 이 cohort에서
source와 domain은 분리 식별되지 않으며(SSOT §20.2), 그 판정은 G5의 identifiability 작업에
속한다. 여기서 말할 수 있는 것은 층별 기술통계가 다르다는 사실뿐이다. 그럼에도 결론 자체는
층 구성을 고정한 stratified 재표집에서도 동일했다(C3).

**유의성에 대하여.** 세 검정 모두 exact p가 underflow한다. `N = 3,835,988`에서 이는 놀라운 일이
아니며 정보량도 크지 않다. 결론을 지지하는 것은 지수의 자릿수가 아니라, 효과 크기가 33%이고
CI가 그 값에 고정되며 tie 전량을 반대편에 세운 보수적 검정과 층을 고정한 재표집 모두에서
결론이 유지된다는 사실의 결합이다.

In [17]:
INTERPRETATION = {
    "median_logTP": median_primary,
    "median_TP_scale": exp_median_primary,
    "median_TP_as_ratio": "4 / 3",
    "median_premium_percent": (exp_median_primary - 1) * 100,
    "P_TP_gt_1": desc["share_TP_gt_1"],
    "per_source": {},
}
for lv in levels:
    sub = Y_C[SRC == lv]
    INTERPRETATION["per_source"][str(lv)] = {
        "n": int(sub.size), "share": float(sub.size / NN),
        "median_logTP": float(np.median(sub)),
        "median_TP": float(np.exp(np.median(sub))),
        "P_TP_gt_1": float((sub > 0).mean()),
        "tie_share": float((sub == 0).mean()),
    }
print(f"중앙 pair premium        {INTERPRETATION['median_premium_percent']:.1f}%  (= 4 대 3)")
print(f"premium 양(+) pair 비율   {INTERPRETATION['P_TP_gt_1']:.2%}")
print("\nsource별 기술통계 (인과·domain 해석 금지):")
for k, v in INTERPRETATION["per_source"].items():
    print(f"  {k}  n={v['n']:,} ({v['share']:.1%})  median TP={v['median_TP']:.4f}  "
          f"P(TP>1)={v['P_TP_gt_1']:.2%}  tie={v['tie_share']:.2%}")
print(f"\npooled median TP = {exp_median_primary:.6f} 는 두 층 어느 쪽 median과도 일치하지 않는다.")

중앙 pair premium        33.3%  (= 4 대 3)
premium 양(+) pair 비율   87.99%

source별 기술통계 (인과·domain 해석 금지):
  025  n=2,485,963 (64.8%)  median TP=1.3158  P(TP>1)=83.36%  tie=7.34%
  026  n=1,350,025 (35.2%)  median TP=1.3617  P(TP>1)=96.50%  tie=1.06%

pooled median TP = 1.333333 는 두 층 어느 쪽 median과도 일치하지 않는다.


## C7 — closeout artifact 기록과 판정

기존 primary result JSON은 덮어쓰지 않는다. closeout 산출물은 별도 파일에 기록한다.

In [18]:
import json as _json, datetime as _dt, platform as _pf, scipy as _sp, duckdb as _dd, pyarrow as _pa

CLOSEOUT = {
    "artifact_id": "NB08_RQ1_SSOT_CLOSEOUT_v001",
    "authority": ["KOEN-TP-RS-001", "RD-SSOT-CANONICAL-RETURN-01"],
    "decision_id": "RD-RQ1-FIRST-RESULT-01",
    "protocol_id": "NB08_RQ1_PROTOCOL_v001",
    "no_primary_protocol_change": True,
    "primary_protocol_cells_modified": 0,
    "evidence_of_record": {
        "primary_result_commit": PRIMARY_RESULT_SHA,
        "primary_result_json": "ssot_nb01/04_NB08_RQ1_RESULTS_v001.json",
        "primary_result_json_sha256": hashlib.sha256(
            (ROOT / "ssot_nb01/04_NB08_RQ1_RESULTS_v001.json").read_bytes()).hexdigest(),
        "decision_commit": "e72274086a7e9c611c9014e6b5612df0e69dae30",
        "cohort_commit": "9b695307c0551be84d4d6c374646bfe001b7b3a9",
        "protocol_commit": "86521fdf04839d2e3e8e5db8e15a08ea067871e3",
        "d04_sha256": actual_sha,
        "pair_set_hash": pair_set_hash,
    },
    "primary_unchanged": {k: v[0] for k, v in guard.items()},
    "CONDITIONAL_NONZERO_SIGN_TEST": CONDITIONAL_NONZERO_SIGN_TEST,
    "TIE_AWARE_MEDIAN_SIGN_ROBUSTNESS": TIE_AWARE,
    "SOURCE_STRATIFIED_BOOTSTRAP_SENSITIVITY": SOURCE_STRATIFIED,
    "REPORTING": REPORTING,
    "CI_DEGENERACY": DEGENERACY,
    "INTERPRETATION": INTERPRETATION,
    "final_claim": ("Under the fixed o200k_base raw-text Track A configuration and the defined final "
                    "paired KO-EN cohort, the pair-level median log tokenization premium was positive."),
    "prohibited_claims": ["causal", "all-tokenizer generalization", "morphology cause",
                          "domain cause", "AI ability", "fixed provider cost claim"],
    "deferred_to_nb11": [
        "B >= 5000 bootstrap (CR-RQ1-BOOTSTRAP-FAST-2000-01은 first release 한정)",
        "source cluster bootstrap (§17.2 level 수 조건 미달)",
        "§17.2의 나머지 CI 대상: geometric mean TP, P(TP>1), median absolute token difference",
        "층 간 이질성의 원인 규명 (G5 identifiability 선행 필요)",
    ],
    "software": {"python": _pf.python_version(), "numpy": np.__version__,
                 "scipy": _sp.__version__, "duckdb": _dd.__version__, "pyarrow": _pa.__version__},
    "created_at_kst": _dt.datetime.now(KST).isoformat(timespec="seconds"),
}

VERDICT = {
    "primary median > 0": median_primary > 0,
    "prespecified signed-rank direction positive": wil_primary["z_normal_approx"] > 0,
    "tie-aware sign robustness same direction": all(
        TIE_AWARE[c]["direction_same_as_primary"] for c in TIE_AWARE),
    "source-stratified does not materially reverse":
        not SOURCE_STRATIFIED["materially_reverses_conclusion"],
    "D04 identity unchanged": actual_sha == D04_SHA256,
}
for k, v in VERDICT.items():
    print(f"  {'OK  ' if v else 'FAIL'} {k}")
CLOSEOUT["verdict_checks"] = VERDICT
CLOSEOUT["verdict"] = ("RQ1_PRIMARY_INFERENCE_PASS / NB08_RQ1_CLOSED" if all(VERDICT.values())
                       else "NB08_RQ1_NOT_CLOSED")

out = ROOT / "ssot_nb01/06_NB08_RQ1_SSOT_CLOSEOUT_v001.json"
out.write_text(_json.dumps(CLOSEOUT, ensure_ascii=False, indent=2, sort_keys=True) + "\n")
print(f"\n기록: {out.relative_to(ROOT)}")
print("\nCI가 동일 lattice value로 degenerate하는 것은 FAIL 조건이 아니다.")
if all(VERDICT.values()):
    print("\nRQ1_PRIMARY_INFERENCE_PASS")
    print("NB08_RQ1_CLOSED")
else:
    print("\nNB08_RQ1_NOT_CLOSED")

  OK   primary median > 0
  OK   prespecified signed-rank direction positive
  OK   tie-aware sign robustness same direction
  OK   source-stratified does not materially reverse
  OK   D04 identity unchanged



기록: ssot_nb01/06_NB08_RQ1_SSOT_CLOSEOUT_v001.json

CI가 동일 lattice value로 degenerate하는 것은 FAIL 조건이 아니다.

RQ1_PRIMARY_INFERENCE_PASS
NB08_RQ1_CLOSED


---

## 최종 claim

허용되는 가장 강한 문장은 다음과 같다.

> Under the fixed `o200k_base` raw-text Track A configuration and the defined final paired KO–EN
> cohort, the pair-level median log tokenization premium was positive.

이 문장은 반드시 effect magnitude와 CI caveat를 동반해 보고한다. 중앙 pair의 premium은
33.3%(= 4 대 3)이며, 95% CI는 관측된 자료에서 동일 lattice 값으로 퇴화한다 — 이는 정밀도가
아니라 결과변수가 정수비 격자 위에 놓여 있다는 사실의 표현이다.

금지되는 진술: causal · all-tokenizer 일반화 · morphology 원인 · domain 원인 · AI ability ·
고정 provider 비용 수치.

RQ1은 여기서 종료한다. 잔여 robustness는 NB11로 이관한다. 다음 canonical stage는 NB06이다.